El script debe dar tratamiento conjuntos de datos grandes

  1.​ Tomar muestras de 5.000 y 10.000 registros.

  2.​ Calcular la matriz de distancias Gower para cada muestra.

  3.​ Medir tiempo de ejecución y uso de memoria.
  
  4.​ Comparar estabilidad de clusters en muestras de distinto tamaño.​

Muestra de 5000 registros:

In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 5000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

Muestra de 10.000 registros:

In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 10000

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante (versión vectorizada) ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

## Paso 4: Comparación de Estabilidad de Clusters

Ahora vamos a comparar la estabilidad de los clusters entre las muestras de 5,000 y 10,000 registros usando AgglomerativeClustering con las matrices de Gower calculadas.


In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Función para calcular matriz de Gower
def calcular_matriz_gower(df_sample):
    """Calcula la matriz de distancias de Gower para una muestra"""
    
    # Variables
    numeric_cols = ['age', 'fnlwgt', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
    ordinal_cols = ['education']
    categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 
                       'race', 'sex', 'native-country', 'income']
    
    # Mapa ordinal
    education_map = {
        'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
        '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
        'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
        'Prof-school': 15, 'Doctorate': 16
    }
    
    # Normalizar education
    df_sample = df_sample.copy()
    df['education_rank'] = df['education'].map(education_map)
    min_rank = df['education_rank'].min()
    max_rank = df['education_rank'].max()
    ordinal_rank_range = max_rank - min_rank
    df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range
    
    # Rangos numéricos
    ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}
    
    # Convertir a arrays
    df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
    df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
    df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)
    
    # Calcular disimilitudes
    numeric_dissim = np.sum([
        np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
        for i, col in enumerate(numeric_cols) if ranges[col] > 0
    ], axis=0)
    
    categorical_dissim = np.sum([
        np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
        for i in range(df_cat.shape[1])
    ], axis=0)
    
    ordinal_dissim = np.sum([
        np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
        for i in range(df_ord.shape[1])
    ], axis=0)
    
    total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim
    
    # Pesos
    is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
    weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)
    
    # Matriz final
    with np.errstate(divide='ignore', invalid='ignore'):
        gower_matrix = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)
    
    return gower_matrix

print("Preparando datos para comparación...")


In [ ]:
# Cargar dataset completo
df = cargar_dataset_adult()

# Crear muestras con la misma semilla para comparación justa
np.random.seed(42)
sample_5k = df.sample(n=5000, random_state=42)
sample_10k = df.sample(n=10000, random_state=42)

print(f"Muestra 5K: {len(sample_5k)} registros")
print(f"Muestra 10K: {len(sample_10k)} registros")

# Calcular matrices de Gower
print("\nCalculando matriz de Gower para muestra de 5,000...")
gower_5k = calcular_matriz_gower(sample_5k)

print("Calculando matriz de Gower para muestra de 10,000...")
gower_10k = calcular_matriz_gower(sample_10k)

print("\nMatrices calculadas correctamente.")


In [ ]:
# Aplicar AgglomerativeClustering con diferentes valores de k
k_values = [3, 4, 5, 6]

results_5k = {}
results_10k = {}

print("Aplicando clustering a muestra de 5,000 registros...")
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = agg.fit_predict(gower_5k)
    silhouette = silhouette_score(gower_5k, labels, metric='precomputed')
    results_5k[k] = {'labels': labels, 'silhouette': silhouette}
    print(f"  k={k}: Silhouette = {silhouette:.4f}")

print("\nAplicando clustering a muestra de 10,000 registros...")
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = agg.fit_predict(gower_10k)
    silhouette = silhouette_score(gower_10k, labels, metric='precomputed')
    results_10k[k] = {'labels': labels, 'silhouette': silhouette}
    print(f"  k={k}: Silhouette = {silhouette:.4f}")

print("\nClustering completado para ambas muestras.")


In [ ]:
# Calcular métricas de comparación
silhouette_5k = [results_5k[k]['silhouette'] for k in k_values]
silhouette_10k = [results_10k[k]['silhouette'] for k in k_values]
differences = [abs(s1 - s2) for s1, s2 in zip(silhouette_5k, silhouette_10k)]

print("Métricas de comparación calculadas.")


In [ ]:
# Análisis detallado de estabilidad
print("="*80)
print("ANÁLISIS DE ESTABILIDAD DE CLUSTERS")
print("="*80)

# Tabla comparativa
comparison_df = pd.DataFrame({
    'k': k_values,
    'Silhouette 5K': [results_5k[k]['silhouette'] for k in k_values],
    'Silhouette 10K': [results_10k[k]['silhouette'] for k in k_values],
    'Diferencia Abs': differences,
    'Diferencia %': [abs(s1 - s2)/s1 * 100 for s1, s2 in zip(silhouette_5k, silhouette_10k)]
})

print("\nTabla comparativa de métricas:")
print(comparison_df.to_string(index=False))

# Estadísticas de estabilidad
mean_diff = np.mean(differences)
max_diff = np.max(differences)
min_diff = np.min(differences)

print(f"\n{'='*80}")
print("ESTADÍSTICAS DE ESTABILIDAD:")
print(f"{'='*80}")
print(f"Diferencia promedio:  {mean_diff:.4f}")
print(f"Diferencia máxima:    {max_diff:.4f} (k={k_values[np.argmax(differences)]})")
print(f"Diferencia mínima:    {min_diff:.4f} (k={k_values[np.argmin(differences)]})")

# Mejor k para cada muestra
best_k_5k = max(results_5k.keys(), key=lambda k: results_5k[k]['silhouette'])
best_k_10k = max(results_10k.keys(), key=lambda k: results_10k[k]['silhouette'])

print(f"\nMejor k para 5K:      k={best_k_5k} (Silhouette={results_5k[best_k_5k]['silhouette']:.4f})")
print(f"Mejor k para 10K:     k={best_k_10k} (Silhouette={results_10k[best_k_10k]['silhouette']:.4f})")

# Conclusión sobre estabilidad
print(f"\n{'='*80}")
print("CONCLUSIONES:")
print(f"{'='*80}")

if best_k_5k == best_k_10k:
    print(f"✓ Ambas muestras convergen al mismo número óptimo de clusters (k={best_k_5k})")
    print("  Esto indica ALTA ESTABILIDAD en la estructura de clustering.")
else:
    print(f"✗ Las muestras difieren en el número óptimo de clusters")
    print(f"  5K sugiere k={best_k_5k}, mientras 10K sugiere k={best_k_10k}")
    print("  Esto indica MENOR ESTABILIDAD.")

if mean_diff < 0.05:
    print(f"\n✓ Diferencia promedio muy baja ({mean_diff:.4f})")
    print("  Los clusters son CONSISTENTES entre muestras de diferente tamaño.")
elif mean_diff < 0.1:
    print(f"\n~ Diferencia promedio moderada ({mean_diff:.4f})")
    print("  Los clusters son RELATIVAMENTE ESTABLES pero hay variación.")
else:
    print(f"\n✗ Diferencia promedio alta ({mean_diff:.4f})")
    print("  Los clusters muestran INESTABILIDAD significativa.")

print(f"\n{'='*80}")


In [ ]:
# Análisis de distribución de tamaños de clusters
print("\nConsistencia en distribuciones de tamaño:")
print("="*60)
for k in k_values:
    sizes_5k = pd.Series(results_5k[k]['labels']).value_counts().sort_index()
    sizes_10k = pd.Series(results_10k[k]['labels']).value_counts().sort_index()
    
    sizes_5k_pct = (sizes_5k / len(sample_5k) * 100)
    sizes_10k_pct = (sizes_10k / len(sample_10k) * 100)
    
    # Mostrar distribuciones
    print(f"\nk={k}:")
    print(f"  Muestra 5K:  {dict(sizes_5k_pct.round(1))}")
    print(f"  Muestra 10K: {dict(sizes_10k_pct.round(1))}")
    
    # Diferencia promedio en porcentajes
    avg_diff = np.mean(np.abs(sizes_5k_pct.values - sizes_10k_pct.values))
    
    print(f"  Diferencia promedio en tamaños: {avg_diff:.2f}%")
    if avg_diff < 5:
        print(f"     ✓ MUY CONSISTENTE")
    elif avg_diff < 10:
        print(f"     ~ MODERADAMENTE CONSISTENTE")
    else:
        print(f"     ✗ POCO CONSISTENTE")


## Conclusiones sobre Estabilidad

Se evaluó la estabilidad comparando muestras de 5,000 y 10,000 registros usando tres criterios: Coeficiente de Silueta, número óptimo de k, y distribución de tamaños de clusters.

Alta estabilidad indica que los clusters representan patrones reales y robustos en los datos. Baja estabilidad sugiere que los clusters son sensibles al tamaño de muestra.
